# DPA Tools Quickstart
Fine-tune a frozen DPA-3.1 descriptor + Ridge regressor on QM9 HOMO–LUMO gap
in under 5 minutes on CPU with just 50 molecules.

Pre-processed data for 50 molecules (40 train / 10 test) is included in `data/`.

## Prerequisites
- Python 3.10+ with `pip install deepmd-kit[dpa-tools]`
- DPA pretrained checkpoint from AIS Square or the DeepModeling release page.
  This demo uses DPA-3.1-3M (`model_branch="Domains_Drug"`).

In [ ]:
import os

# Force CPU mode — avoids device-mismatch errors when the checkpoint
# was saved with CUDA tensors.  Remove this line if you have a GPU and
# want to use it (may require additional setup).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

MODEL_PATH = os.environ.get("DPA_MODEL_PATH", "/share/DPA-3.1-3M.pt")
print(f"Using model: {MODEL_PATH}")

## Step 1 — Load model
`frozen_sklearn` freezes the DPA backbone, extracts descriptors once, and fits
a scikit-learn Ridge regressor. No GPU needed.

In [ ]:
from deepmd.dpa_tools import DPAFineTuner
from pathlib import Path
import numpy as np

HERE = Path().resolve()
TRAIN_DIR = HERE / "data" / "train"
TEST_DIR  = HERE / "data" / "test"

model = DPAFineTuner(
    pretrained=MODEL_PATH,
    model_branch="Domains_Drug",
    strategy="frozen_sklearn",
    predictor="linear",
    pooling="mean",
    seed=42,
)

## Step 2 — Fit

The data directory contains one `sys_*/` sub-directory per molecule.
We use a glob pattern so that each sub-directory is loaded as a separate
deepmd/npy system.  Labels are read from `set.000/gap.npy` inside each system.

In [ ]:
model.fit(train_data=str(TRAIN_DIR) + "/*", target_key="gap")

## Step 3 — Evaluate on held-out test set

In [ ]:
metrics = model.evaluate(data=str(TEST_DIR) + "/*")
print(f"MAE  : {metrics.mae:.4f} eV")
print(f"R²   : {metrics.r2:.4f}")
print(f"RMSE : {metrics.rmse:.4f} eV")
print(f"N    : {metrics.predictions.shape[0]}")

## Step 4 — Freeze and reload
Save a portable bundle and reload it with `DPAPredictor` (no training dependencies).

In [ ]:
model.freeze("frozen_model.pth")

from deepmd.dpa_tools import DPAPredictor
pred = DPAPredictor("frozen_model.pth")
result = pred.predict(str(TEST_DIR) + "/*")
print(f"Predictions shape: {result.predictions.shape}")

## Next steps
- Other strategies (`linear_probe`, `finetune`, `mft`) are documented in
  [`../README.md`](../README.md).
- To regenerate the demo data from raw GDB9, run `scripts/prepare_data.py`.
- To use your own data, replace `TRAIN_DIR` / `TEST_DIR` with your own
  deepmd/npy directories and set `target_key` to match your label key.